# Construcción de la Master Table (Sprint 1)

Pipeline para procesar las fuentes crudas y consolidar la tabla analítica a nivel de cliente único (`customer_unique_id`).

## 1. Importación de Librerías

In [18]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Configuración de Rutas

In [19]:
PROJECT_ROOT = Path("../..").resolve()

PROFILE_SOURCE = os.getenv("PROFILE_SOURCE", "dev").strip().lower()  # raw | dev
if PROFILE_SOURCE not in {"raw", "dev"}:
    raise ValueError("PROFILE_SOURCE debe ser 'raw' o 'dev'")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SPLIT_DEV_DIR = PROJECT_ROOT / "data" / "splits" / "temporal_2018q4" / "dev"
PARQUET_DIR = RAW_DIR if PROFILE_SOURCE == "raw" else SPLIT_DEV_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Profile source:", PROFILE_SOURCE)
print("Raw dir:", RAW_DIR)
print("Parquet dir:", PARQUET_DIR)
print("Processed dir:", PROCESSED_DIR)


Project root: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum
Profile source: dev
Raw dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw
Parquet dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/splits/temporal_2018q4/dev
Processed dir: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/processed


## 3. Carga de Datos

In [20]:
def load_dataset(file_stem: str, force_raw: bool = False) -> pd.DataFrame:
    """Load from the selected source while preserving raw catalogs when needed."""
    source_dir = RAW_DIR if force_raw else PARQUET_DIR

    split_name_map = {
        "olist_orders_dataset": "orders",
        "olist_order_items_dataset": "order_items",
        "olist_order_payments_dataset": "order_payments",
        "olist_order_reviews_dataset": "order_reviews",
    }

    effective_stem = file_stem
    if (not force_raw) and PROFILE_SOURCE == "dev":
        effective_stem = split_name_map.get(file_stem, file_stem)

    parquet_path = source_dir / f"{effective_stem}.parquet"
    csv_path = RAW_DIR / f"{file_stem}.csv"

    if parquet_path.exists():
        print(f"Loading Parquet: {parquet_path}")
        return pd.read_parquet(parquet_path)

    if force_raw and csv_path.exists():
        print(f"Loading CSV: {csv_path}")
        return pd.read_csv(csv_path)

    raise FileNotFoundError(
        f"Dataset not found: {file_stem} (effective: {effective_stem}) in {source_dir}"
    )

# Los hechos transaccionales cambian con raw/dev; los catálogos siguen viniendo de raw.
customers = load_dataset("olist_customers_dataset", force_raw=True)
orders = load_dataset("olist_orders_dataset")
payments = load_dataset("olist_order_payments_dataset")
reviews = load_dataset("olist_order_reviews_dataset")
order_items = load_dataset("olist_order_items_dataset")
products = load_dataset("olist_products_dataset", force_raw=True)
sellers = load_dataset("olist_sellers_dataset", force_raw=True)


Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw/olist_customers_dataset.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/splits/temporal_2018q4/dev/orders.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/splits/temporal_2018q4/dev/order_payments.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/splits/temporal_2018q4/dev/order_reviews.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/splits/temporal_2018q4/dev/order_items.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw/olist_products_dataset.parquet
Loading Parquet: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/raw/olist_sellers_dataset.parquet


## 4. Dimensiones y Duplicados Crudos

In [21]:
datasets = {
    "customers": customers,
    "orders": orders,
    "payments": payments,
    "reviews": reviews,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
}

summary = []

for name, df in datasets.items():
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_nulls": df.isna().sum().sum(),
    })

pd.DataFrame(summary)


,dataset,rows,columns,duplicated_rows,total_nulls
0,customers,99441,5,0,0
1,orders,92909,8,0,4522
2,payments,97168,5,0,0
3,reviews,92718,7,0,137722
4,order_items,105401,7,0,0
5,products,32951,9,0,2448
6,sellers,3095,4,0,0


## 5. Conversión de Fechas

In [22]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].head()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
3,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
4,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01


## 6. Métricas de Pago por Pedido

In [23]:
order_payments = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_methods_count=("payment_type", "nunique"),
        main_payment_type=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

order_payments.head()


,order_id,order_payment_value,payment_installments,payment_methods_count,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,credit_card
3,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,credit_card
4,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,1,1,boleto


## 7. Métricas de Artículos por Pedido

In [24]:
order_item_features = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_items_count=("order_item_id", "count"),
        order_products_count=("product_id", "nunique"),
        order_price_total=("price", "sum"),
        order_freight_total=("freight_value", "sum"),
        sellers_count=("seller_id", "nunique"),
    )
)

order_item_features["freight_ratio"] = (
    order_item_features["order_freight_total"] /
    (order_item_features["order_price_total"] + order_item_features["order_freight_total"])
).replace([np.inf, -np.inf], np.nan)

order_item_features.head()


,order_id,order_items_count,order_products_count,order_price_total,order_freight_total,sellers_count,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.9,13.29,1,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.9,19.93,1,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,1,199.0,17.87,1,0.082400
3,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.9,18.14,1,0.083196
4,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,21.9,12.69,1,0.366869


## 8. Calificación de Reseñas por Pedido

In [25]:
order_reviews = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique"),
    )
)

order_reviews.head()


,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1
4,00048cc3ae777c65dbb7d2a0634bc1ea,4.0,1


## 9. Consolidación a Nivel Pedido

In [26]:
orders_enriched = (
    orders
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner")
    .merge(order_payments, on="order_id", how="left")
    .merge(order_item_features, on="order_id", how="left")
    .merge(order_reviews, on="order_id", how="left")
)

orders_enriched["delivery_days"] = (
    orders_enriched["order_delivered_customer_date"] -
    orders_enriched["order_purchase_timestamp"]
).dt.days

orders_enriched["estimated_delivery_days"] = (
    orders_enriched["order_estimated_delivery_date"] -
    orders_enriched["order_purchase_timestamp"]
).dt.days

orders_enriched["is_delivered"] = np.where(
    orders_enriched["order_status"].eq("delivered"),
    1,
    0,
)

orders_enriched["is_canceled"] = np.where(
    orders_enriched["order_status"].eq("canceled"),
    1,
    0,
)

orders_enriched["is_late_delivery"] = np.where(
    orders_enriched["order_delivered_customer_date"] > orders_enriched["order_estimated_delivery_date"],
    1,
    0,
)

orders_enriched.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,order_payment_value,payment_installments,payment_methods_count,main_payment_type,order_items_count,order_products_count,order_price_total,order_freight_total,sellers_count,freight_ratio,review_score,review_count,delivery_days,estimated_delivery_days,is_delivered,is_canceled,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,38.71,1.0,2.0,voucher,1.0,1.0,29.99,8.72,1.0,0.225265,4.0,1.0,8.0,15,1,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,141.46,1.0,1.0,boleto,1.0,1.0,118.70,22.76,1.0,0.160894,4.0,1.0,13.0,19,1,0,0
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,72.20,1.0,1.0,credit_card,1.0,1.0,45.00,27.20,1.0,0.376731,5.0,1.0,13.0,26,1,0,0
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,28.62,1.0,1.0,credit_card,1.0,1.0,19.90,8.72,1.0,0.304682,5.0,1.0,2.0,12,1,0,0
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,80bb27c7c16e8f973207a5086ab329e2,175.26,6.0,1.0,credit_card,1.0,1.0,147.90,27.36,1.0,0.156111,4.0,1.0,16.0,22,1,0,0


## 10. Agregación a Nivel Cliente

Consolidación de comportamiento de compra general e histórico financiero.

In [27]:
reference_date = orders_enriched["order_purchase_timestamp"].max()

# 1. Métricas financieras basadas únicamente en pedidos entregados (facturación neta real)
customer_financials = (
    orders_enriched[orders_enriched["order_status"].eq("delivered")]
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_spent=("order_payment_value", "sum"),
        avg_ticket=("order_payment_value", "mean"),
        avg_order_price=("order_price_total", "mean"),
        avg_freight_value=("order_freight_total", "mean"),
        avg_freight_ratio=("freight_ratio", "mean"),
    )
)

# 2. Métricas de volumen y operacionales basadas en todas las órdenes (incluyendo canceladas)
customer_general = (
    orders_enriched
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("order_items_count", "sum"),
        total_products=("order_products_count", "sum"),
        avg_review_score=("review_score", "mean"),
        total_reviews=("review_count", "sum"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_estimated_delivery_days=("estimated_delivery_days", "mean"),
        delivered_orders=("is_delivered", "sum"),
        canceled_orders=("is_canceled", "sum"),
        late_deliveries=("is_late_delivery", "sum"),
        payment_methods_count=("payment_methods_count", "max"),
        max_payment_installments=("payment_installments", "max"),
        avg_payment_installments=("payment_installments", "mean"),
        main_payment_type=("main_payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
)

# 3. Unir métricas operacionales y financieras
customer_features = customer_general.merge(customer_financials, on="customer_unique_id", how="left")

customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"] - customer_features["first_purchase"]
).dt.days

customer_features["cancellation_rate"] = (
    customer_features["canceled_orders"] / customer_features["total_orders"]
)

customer_features["late_delivery_rate"] = (
    customer_features["late_deliveries"] / customer_features["total_orders"]
)

customer_features.head()

,customer_unique_id,total_orders,total_items,total_products,avg_review_score,total_reviews,first_purchase,last_purchase,avg_delivery_days,avg_estimated_delivery_days,delivered_orders,canceled_orders,late_deliveries,payment_methods_count,max_payment_installments,avg_payment_installments,main_payment_type,total_spent,avg_ticket,avg_order_price,avg_freight_value,avg_freight_ratio,recency_days,customer_lifetime_days,cancellation_rate,late_delivery_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1.0,1.0,5.0,1.0,2018-05-10 10:56:27,2018-05-10 10:56:27,6.0,10.0,1,0,0,1.0,8.0,8.0,credit_card,141.90,141.90,129.90,12.00,0.084567,82,0,0.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1.0,1.0,4.0,1.0,2018-05-07 11:11:27,2018-05-07 11:11:27,3.0,7.0,1,0,0,1.0,1.0,1.0,credit_card,27.19,27.19,18.90,8.29,0.304892,85,0,0.0,0.0
2,0000f46a3911fa3c0805444483337064,1,1.0,1.0,3.0,1.0,2017-03-10 21:05:03,2017-03-10 21:05:03,25.0,27.0,1,0,0,1.0,8.0,8.0,credit_card,86.22,86.22,69.00,17.22,0.199722,508,0,0.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1.0,1.0,4.0,1.0,2017-10-12 20:29:41,2017-10-12 20:29:41,20.0,31.0,1,0,0,1.0,4.0,4.0,credit_card,43.62,43.62,25.99,17.63,0.404172,292,0,0.0,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,1.0,1.0,5.0,1.0,2017-11-14 19:45:42,2017-11-14 19:45:42,13.0,20.0,1,0,0,1.0,6.0,6.0,credit_card,196.89,196.89,180.00,16.89,0.085784,259,0,0.0,0.0


## 10b. Características del Catálogo de Productos por Cliente

Precio máximo e individual promedio de ítems comprados, y categoría principal del cliente.
Estas features capturan el **nivel de precio** de los artículos (independiente del gasto total)
y son las más predictivas para identificar clientes premium.

In [28]:
customer_catalog = (
    order_items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(orders[["order_id", "customer_id"]], on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .groupby("customer_unique_id", as_index=False)
    .agg(
        max_item_price=("price", "max"),
        avg_item_price=("price", "mean"),
        top_category=(
            "product_category_name",
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
        ),
    )
)

print(f"Clientes con features de catálogo: {len(customer_catalog)}")
customer_catalog.head()

Clientes con features de catálogo: 89182


,customer_unique_id,max_item_price,avg_item_price,top_category
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,129.90,cama_mesa_banho
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,18.90,beleza_saude
2,0000f46a3911fa3c0805444483337064,69.00,69.00,papelaria
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,25.99,telefonia
4,0004aac84e0df4da2b147fca70cf8255,180.00,180.00,telefonia


## 11. Integración de Datos Geográficos

In [29]:
customer_geo = (
    customers[
        [
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
        ]
    ]
    .drop_duplicates(subset=["customer_unique_id"], keep="last")
)

master_table = (
    customer_geo
    .merge(customer_features, on="customer_unique_id", how="left")
    .merge(customer_catalog, on="customer_unique_id", how="left")
)

# Imputar con 0 solo las columnas cuantitativas de conteo/suma
cols_to_fill_zero = [
    "total_spent",
    "total_orders",
    "total_items",
    "total_products",
    "total_reviews",
    "delivered_orders",
    "canceled_orders",
    "late_deliveries",
    "cancellation_rate",
    "late_delivery_rate",
]
master_table[cols_to_fill_zero] = master_table[cols_to_fill_zero].fillna(0)

master_table.head()

,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_orders,total_items,total_products,avg_review_score,total_reviews,first_purchase,last_purchase,avg_delivery_days,avg_estimated_delivery_days,delivered_orders,canceled_orders,late_deliveries,payment_methods_count,max_payment_installments,avg_payment_installments,main_payment_type,total_spent,avg_ticket,avg_order_price,avg_freight_value,avg_freight_ratio,recency_days,customer_lifetime_days,cancellation_rate,late_delivery_rate,max_item_price,avg_item_price,top_category
0,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1.0,1.0,1.0,4.0,1.0,2017-05-16 15:05:35,2017-05-16 15:05:35,8.0,19.0,1.0,0.0,0.0,1.0,2.0,2.0,credit_card,146.87,146.87,124.99,21.88,0.148975,441.0,0.0,0.0,0.0,124.99,124.99,moveis_escritorio
1,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1.0,1.0,1.0,5.0,1.0,2018-01-12 20:48:24,2018-01-12 20:48:24,16.0,24.0,1.0,0.0,0.0,1.0,8.0,8.0,credit_card,335.48,335.48,289.00,46.48,0.138548,200.0,0.0,0.0,0.0,289.00,289.00,utilidades_domesticas
2,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1.0,1.0,1.0,5.0,1.0,2018-05-19 16:07:45,2018-05-19 16:07:45,26.0,24.0,1.0,0.0,1.0,1.0,7.0,7.0,credit_card,157.73,157.73,139.94,17.79,0.112788,73.0,0.0,0.0,1.0,139.94,139.94,moveis_escritorio
3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,1.0,1.0,1.0,5.0,1.0,2018-03-13 16:06:38,2018-03-13 16:06:38,14.0,27.0,1.0,0.0,0.0,1.0,1.0,1.0,credit_card,173.30,173.30,149.94,23.36,0.134795,140.0,0.0,0.0,0.0,149.94,149.94,moveis_escritorio
4,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,1.0,1.0,1.0,5.0,1.0,2018-07-29 09:51:30,2018-07-29 09:51:30,11.0,16.0,1.0,0.0,0.0,1.0,8.0,8.0,credit_card,252.25,252.25,230.00,22.25,0.088206,2.0,0.0,0.0,0.0,230.00,230.00,casa_conforto


## 12. Generación del Target `is_premium`

Definido preliminarmente en el percentil 80 de gasto neto.

In [30]:
premium_threshold = master_table["total_spent"].quantile(0.80)

master_table["is_premium"] = np.where(
    master_table["total_spent"] >= premium_threshold,
    1,
    0,
)

print("Premium threshold:", premium_threshold)

master_table["is_premium"].value_counts(normalize=True).rename("proportion")


Premium threshold: 197.01


is_premium
0    0.799992
1    0.200008
Name: proportion, dtype: float64

## 13. Validación de Calidad de la Master Table

In [31]:
print("Rows:", master_table.shape[0])
print("Columns:", master_table.shape[1])
print("Duplicated customer_unique_id:", master_table["customer_unique_id"].duplicated().sum())
print("Null values in key columns:", master_table[["customer_unique_id", "total_spent", "total_orders"]].isna().sum().sum())
print("Null values total:", master_table.isna().sum().sum())

master_table.describe(include="all").T.head(30)


Rows: 96096
Columns: 33
Duplicated customer_unique_id: 0
Null values in key columns: 0
Null values total: 129909


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_unique_id,96096,96096,861eff4711a542e4b93843c6dd7febb0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,96096.0,NaN,NaN,NaN,35184.412463,1003.0,11390.0,24440.0,59032.75,99990.0,29800.101792
customer_city,96096,4119,sao paulo,14971,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,96096,27,SP,40292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_orders,96096.0,NaN,NaN,NaN,0.966835,0.0,1.0,1.0,1.0,14.0,0.327031
total_items,96096.0,NaN,NaN,NaN,1.09683,0.0,1.0,1.0,1.0,21.0,0.671931
total_products,96096.0,NaN,NaN,NaN,0.995837,0.0,1.0,1.0,1.0,13.0,0.417853
avg_review_score,89130.0,NaN,NaN,NaN,4.072735,1.0,4.0,5.0,5.0,5.0,1.349543
total_reviews,96096.0,NaN,NaN,NaN,0.964848,0.0,1.0,1.0,1.0,14.0,0.364444
first_purchase,89819,NaN,NaN,NaN,2017-12-15 02:17:04.741123840,2016-09-04 21:15:19,2017-08-30 18:19:20,2018-01-05 17:03:43,2018-04-14 08:30:40,2018-07-31 23:54:20,NaN


## 14. Métricas Descriptivas de Negocio

In [32]:
business_metrics = master_table.groupby("is_premium").agg(
    customers=("customer_unique_id", "count"),
    avg_total_spent=("total_spent", "mean"),
    median_total_spent=("total_spent", "median"),
    avg_total_orders=("total_orders", "mean"),
    avg_ticket=("avg_ticket", "mean"),
    avg_review_score=("avg_review_score", "mean"),
    avg_recency_days=("recency_days", "mean"),
)

business_metrics


,customers,avg_total_spent,median_total_spent,avg_total_orders,avg_ticket,avg_review_score,avg_recency_days
is_premium,,,,,,,
0,76876,82.578643,76.17,0.935025,92.274307,4.082653,227.311421
1,19220,420.849725,302.55,1.094069,402.660473,4.036274,221.161238


## 15. Exportación a Parquet

In [33]:
output_name = "master_table.parquet" if PROFILE_SOURCE == "raw" else "master_table_dev.parquet"
output_file = PROCESSED_DIR / output_name

master_table.to_parquet(
    output_file,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Master Table saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")


Master Table saved to: /var/www/codigo/maestria_ia/umsa/mod_13/miadas_mod_13_scrum/data/processed/master_table_dev.parquet
File size: 7.83 MB


## 16. Variables Generadas

In [34]:
master_table.columns.tolist()

['customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_orders',
 'total_items',
 'total_products',
 'avg_review_score',
 'total_reviews',
 'first_purchase',
 'last_purchase',
 'avg_delivery_days',
 'avg_estimated_delivery_days',
 'delivered_orders',
 'canceled_orders',
 'late_deliveries',
 'payment_methods_count',
 'max_payment_installments',
 'avg_payment_installments',
 'main_payment_type',
 'total_spent',
 'avg_ticket',
 'avg_order_price',
 'avg_freight_value',
 'avg_freight_ratio',
 'recency_days',
 'customer_lifetime_days',
 'cancellation_rate',
 'late_delivery_rate',
 'max_item_price',
 'avg_item_price',
 'top_category',
 'is_premium']